# Anonymous pairwise evaluation


In [ ]:
from pathlib import Path
import sys
import json
import shutil

REPO_ROOT = Path('/kaggle/working/PA2')
assert (REPO_ROOT / 'evaluate_pairwise.py').is_file(), (
    'Place your completed repository at /kaggle/working/PA2 before running this notebook.'
)
%cd /kaggle/working/PA2
sys.path.insert(0, str(REPO_ROOT))

from evaluate_pairwise import (
    build_pairwise_comparisons, randomize_response_order,
    write_pairwise_comparisons, judge_pairwise_batches, summarize_judgments,
)
GENERATION_OUTPUTS_OVERRIDE = Path('/kaggle/working/PA2/outputs/eval')

def find_generation_outputs():
    if GENERATION_OUTPUTS_OVERRIDE is not None:
        root = Path(GENERATION_OUTPUTS_OVERRIDE)
        candidate_files = list(root.rglob('alpaca_sft.json')) if root.exists() else []
        for file_path in candidate_files:
            folder = file_path.parent
            if (folder / 'alpaca_dpo_ultrafeedback.json').is_file() and (folder / 'hh_sft.json').is_file():
                return folder
        raise FileNotFoundError(f'Could not find all generation JSON files under {root}')

    input_root = Path('/kaggle/input')
    if input_root.exists():
        for file_path in input_root.rglob('alpaca_sft.json'):
            folder = file_path.parent
            if (folder / 'alpaca_base.json').is_file() and (folder / 'alpaca_dpo_ultrafeedback.json').is_file() and (folder / 'hh_sft.json').is_file():
                return folder
        for archive_path in input_root.rglob('generation_outputs.zip'):
            extract_root = Path('/kaggle/working/attached_generation_outputs')
            extract_root.mkdir(parents=True, exist_ok=True)
            shutil.unpack_archive(str(archive_path), str(extract_root))
            candidates = list(extract_root.rglob('alpaca_sft.json'))
            for file_path in candidates:
                folder = file_path.parent
                if (folder / 'alpaca_base.json').is_file() and (folder / 'alpaca_dpo_ultrafeedback.json').is_file() and (folder / 'hh_sft.json').is_file():
                    return folder
    raise FileNotFoundError(
        'Generation outputs were not found. Attach the generation notebook output dataset, ' 
        'or set GENERATION_OUTPUTS_OVERRIDE to the directory containing alpaca_sft.json.'
    )

GENERATION_OUTPUTS = find_generation_outputs()
print('Using generation files:', GENERATION_OUTPUTS)
RESULTS = REPO_ROOT / 'outputs' / 'evaluation'
RESULTS.mkdir(parents=True, exist_ok=True)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def require_generation_files(*names):
    missing = [name for name in names if not (GENERATION_OUTPUTS / name).is_file()]
    if missing:
        raise FileNotFoundError(f'Missing generation files: {missing}')

require_generation_files(
    'alpaca_base.json', 'alpaca_sft.json', 'alpaca_dpo_ultrafeedback.json',
    'hh_sft.json', 'hh_dpo.json', 'persona_base_eval.json', 'persona_lora_dpo_eval.json'
)


## Build anonymous pair files

All pair files are written before any judge requests. The first comparison evaluates the baseline-to-SFT change; the next two evaluate each DPO run against SFT on its assigned set; the final comparison checks the persona adapter against its base model.


In [ ]:
comparison_specs = [
    (
        'baseline_vs_sft', 'alpaca_base.json', 'alpaca_sft.json',
        'Qwen2.5-0.5B base', 'SFT', 42,
    ),
    (
        'ultrafeedback_dpo_vs_sft', 'alpaca_sft.json', 'alpaca_dpo_ultrafeedback.json',
        'SFT', 'UltraFeedback DPO', 43,
    ),
    (
        'hh_dpo_vs_sft', 'hh_sft.json', 'hh_dpo.json',
        'SFT', 'HH-RLHF DPO', 44,
    ),
    (
        'persona_lora_dpo_vs_base', 'persona_base_eval.json', 'persona_lora_dpo_eval.json',
        'SmolLM2-360M-Instruct base', 'Persona LoRA DPO', 45,
    ),
]

pair_sets = {}
for set_name, left_file, right_file, left_name, right_name, seed in comparison_specs:
    pairs = build_pairwise_comparisons(
        read_json(GENERATION_OUTPUTS / left_file), read_json(GENERATION_OUTPUTS / right_file),
        sft_name=left_name, dpo_name=right_name,
    )
    pairs = randomize_response_order(pairs, seed=seed)
    pair_path = RESULTS / f'{set_name}_pairs.jsonl'
    write_pairwise_comparisons(pairs, pair_path)
    pair_sets[set_name] = pairs
    print('Saved', pair_path.name, 'rows=', len(pairs))


## Judge with an available Kaggle model



In [ ]:
import kaggle_benchmarks as kbench

available_models = list(kbench.llms.keys())
print('Available judge models:', available_models)
preferred_judges = [
    'google/gemini-3-flash-preview',
    'google/gemini-3.5-flash',
    'google/gemini-2.5-flash',
    'google/gemini-3.1-flash-lite-preview',
]
JUDGE_MODEL = next((name for name in preferred_judges if name in available_models), available_models[0] if available_models else None)
if JUDGE_MODEL is None:
    raise RuntimeError(
        'None of the preferred judge models is available. Set JUDGE_MODEL to one of the names printed above.'
    )
print('Selected judge:', JUDGE_MODEL)
judge_llm = kbench.llms[JUDGE_MODEL]

def kaggle_judge(prompt):
    return judge_llm.prompt(prompt)

summary_records = []
for set_name, *_ in comparison_specs:
    pairs = pair_sets[set_name]
    output_name = f'{set_name}_judged.jsonl'
    judgments = judge_pairwise_batches(
        pairs, kaggle_judge, RESULTS / output_name,
        batch_size=5, sleep_seconds=5, max_retries=3, reuse_judgments=True,
    )
    challenger = next(spec[4] for spec in comparison_specs if spec[0] == set_name)
    raw_summary = summarize_judgments(judgments, dpo_name=challenger)
    summary = {
        'comparison': set_name, 'challenger': challenger, 'judge_model': JUDGE_MODEL,
        'total_pairs': raw_summary['total'],
        'challenger_wins': raw_summary['dpo_wins'],
        'opponent_wins': raw_summary['dpo_losses'],
        'ties': raw_summary['ties'], 'unknown': raw_summary['unknown'],
        'failures': raw_summary['failures'],
        'win_rate_excluding_ties': raw_summary['win_rate_excluding_ties'],
        'win_rate_including_ties': raw_summary['win_rate_including_ties'],
        'unknown_rate': raw_summary['unknown_rate'], 'failure_rate': raw_summary['failure_rate'],
    }
    summary_path = RESULTS / f'{set_name}_summary.json'
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    summary_records.append(summary)
    print(json.dumps(summary, indent=2))


## Summary plot and archive



In [ ]:
import matplotlib.pyplot as plt

if summary_records:
    labels = [item['comparison'].replace('_', '\n') for item in summary_records]
    x = list(range(len(summary_records)))
    width = 0.24
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar([p - width for p in x], [item['challenger_wins'] for item in summary_records], width=width, label='Challenger wins')
    ax.bar(x, [item['ties'] for item in summary_records], width=width, label='Ties')
    ax.bar([p + width for p in x], [item['opponent_wins'] for item in summary_records], width=width, label='Opponent wins')
    ax.set_xticks(x, labels)
    ax.set_ylabel('Judged pairs')
    ax.set_title('Anonymous pairwise judgments')
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS / 'pairwise_summary.png', dpi=160)
    plt.show()
    plt.close(fig)

(RESULTS / 'evaluation_manifest.json').write_text(
    json.dumps({'judge_model': JUDGE_MODEL, 'comparisons': summary_records}, indent=2),
    encoding='utf-8',
)
archive = shutil.make_archive('/kaggle/working/evaluation_outputs', 'zip', root_dir=RESULTS)
print('Evaluation results:', RESULTS)
print('Archive:', archive)
